# Uinta Mountains: Forest Change, Land Use, and Disturbance 1985–2024
### Using USFS Landscape Change Monitoring System (LCMS) in Google Earth Engine

---

**Research Question:** How have forest cover and land use changed in the Uinta Mountains in relation to wildfire and insect/disease disturbance from the 1980s through today?

**Dataset:** [LCMS v2024-10](https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_LCMS_v2024-10) — USFS/GTAC annual land cover, land use, and change maps at 30 m, 1985–2024 (CONUS + SE Alaska).

LCMS produces three annual thematic products:
| Band | What it shows |
|------|--------------|
| `Land_Cover` | What is on the ground (Trees, Shrubs, Grass, Barren, Water, etc.) |
| `Land_Use` | How the land is used (Forest, Agriculture, Developed, Rangeland, etc.) |
| `Change` | What changed and how (Wildfire, Insect/Disease, Tree Removal, Successional Growth, Stable, etc.) |

**Prerequisites**
- A Google Earth Engine (GEE) account — [sign up here](https://earthengine.google.com/signup/)
- Python ≥ 3.9 with `earthengine-api` and `geeViz` installed:
  ```bash
  pip install earthengine-api geeViz
  ```

> 💡 **Workshop note:** This notebook is designed to run sequentially top-to-bottom. All cells are self-documenting. Run `Kernel → Restart & Run All` for a clean start.


## 1 · Setup — Imports and Authentication

In [ ]:
import ee
import geeViz.geeView as gv
import geeViz.getSummaryAreasLib as sal
from geeViz.outputLib import charts as cl
from IPython.display import display, HTML

# ── Authentication ────────────────────────────────────────────────────────────
# Run this once per machine/account to store credentials locally.
# After that, comment it out and just call ee.Initialize() below.
# ee.Authenticate()

# ── Initialization ────────────────────────────────────────────────────────────
# Replace 'your-project-id' with your GEE Cloud project ID.
ee.Initialize(project='your-project-id')

# geeViz Map object (singleton — do not call gv.Map())
Map = gv.Map
print('Earth Engine initialized successfully.')
print('geeViz version:', gv.__version__)


## 2 · Study Area — The Uinta Mountains

The [Uinta Mountains](https://en.wikipedia.org/wiki/Uinta_Mountains) are a unique east–west trending range in northeastern Utah and southwestern Wyoming, reaching over 13,500 ft (Kings Peak). They are almost entirely publicly managed (Ashley and Wasatch-Cache National Forests, plus the High Uintas Wilderness) and have experienced significant wildfire activity, spruce beetle outbreaks, and mixed conifer disturbance since the 1980s.

The bounding box below covers the core of the range. You can swap it for a more precise National Forest boundary (code shown in comments).

In [ ]:
# ── Option A: Bounding box (fast, no EE server call needed) ───────────────────
# Covers the Uinta Mountains from the Wasatch front to the UT/CO border.
study_area = ee.Geometry.BBox(-111.2, 40.2, -109.5, 41.1)

# ── Option B: National Forest boundaries (more ecologically precise) ──────────
# Uncomment to use the Ashley + Wasatch-Cache NF boundaries instead.
# ashley_nf      = sal.getUSFSForests(forest_name='Ashley')
# wasatch_nf     = sal.getUSFSForests(forest_name='Wasatch-Cache')
# nf_combined    = ashley_nf.merge(wasatch_nf)
# study_area     = nf_combined.geometry().convexHull(maxError=500)

print('Study area type:', study_area.getInfo()['type'])

# Quick area estimate
area_km2 = study_area.area(maxError=500).divide(1e6).getInfo()
print(f'Approximate area: {area_km2:,.0f} km²')


## 3 · Load LCMS Data

LCMS v2024-10 covers 1985–2024 (40 years). Each image in the collection represents one year. We filter spatially to the study area first, then inspect the collection.

In [ ]:
LCMS_ASSET = 'USFS/GTAC/LCMS/v2024-10'

# Filter to study area
lcms = ee.ImageCollection(LCMS_ASSET).filterBounds(study_area)

# Basic metadata
n_images = lcms.size().getInfo()
years    = lcms.aggregate_array('year').distinct().sort().getInfo()
bands    = lcms.first().bandNames().getInfo()

print(f'Images in collection : {n_images}')
print(f'Years                : {years[0]}–{years[-1]}')
print(f'Bands                : {bands}')


## 4 · Interactive Map — Land Cover, Land Use, and Change

The interactive geeViz map lets you explore the three LCMS layers for the most recent year (2024). You can:
- **Toggle layers** on/off with the checkboxes in the layer panel
- **Click any pixel** to query its class value
- **Draw a polygon** and click "Chart Selected Area" to get class breakdowns for sub-regions

> 🗺️ The map opens in your browser (or inline in JupyterLab). Use the layer panel on the right to switch between Land Cover, Land Use, and Change views.

In [ ]:
Map.clearMap()

# ── Most recent year as single mosaic ─────────────────────────────────────────
LATEST_YEAR = 2024
lcms_2024 = lcms.filter(ee.Filter.eq('year', LATEST_YEAR)).mosaic()

# Land Cover (what's on the ground)
Map.addLayer(
    lcms_2024.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    f'Land Cover {LATEST_YEAR}',
    True,
)

# Land Use (how land is used/managed)
Map.addLayer(
    lcms_2024.select('Land_Use'),
    {'autoViz': True, 'canAreaChart': True},
    f'Land Use {LATEST_YEAR}',
    False,
)

# Change (what disturbance or successional process occurred)
Map.addLayer(
    lcms_2024.select('Change'),
    {'autoViz': True, 'canAreaChart': True},
    f'Change {LATEST_YEAR}',
    False,
)

# ── Study area outline ────────────────────────────────────────────────────────
Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffffff', 'strokeWidth': 2,
     'fillColor': '00000000'},
    'Uinta Mountains Study Area',
)

Map.setCenter(-110.35, 40.65, 9)
Map.view()  # opens the interactive map


## 5 · Analysis — How Has Land Cover Changed Over Time?

We compute the annual percentage of each Land Cover class across the study area from 1985 through 2024. The stacked area chart makes it easy to see which cover types have gained or lost ground, and when the largest shifts occurred.

In [ ]:
lc_result = cl.summarize_and_chart(
    lcms,
    geometry=study_area,
    band_names='Land_Cover',
    scale=120,              # 120 m — fast enough for annual 40-year analysis
    area_format='Percentage',
    title='Uinta Mountains — Annual Land Cover 1985–2024',
    chart_type='line',
    stacked=True,
    date_format='YYYY',
    width=1000,
    height=550,
)

# Save to HTML for sharing / embedding
cl.save_chart_html(lc_result['chart'], 'uinta_land_cover_timeseries.html')
print('Chart saved to uinta_land_cover_timeseries.html')

# Display inline
lc_result['chart'].show()


### 5a · Inspect the underlying data

The `summarize_and_chart()` function returns both a chart and the raw DataFrame. Use the DataFrame to look up specific values, run statistics, or build custom plots.

In [ ]:
# DataFrame of annual Land Cover percentages
lc_df = lc_result['df']
print(lc_df.to_markdown())


## 6 · Analysis — How Have Disturbance Agents Changed Over Time?

The LCMS `Change` band tracks the dominant change process for each pixel each year. For the Uinta Mountains, the most ecologically significant change classes are:

| Class | Value | Ecological Significance |
|-------|-------|------------------------|
| **Wildfire** | 7 | Crown fires, surface fires — immediate canopy loss |
| **Insect, Disease, or Drought Stress** | 12 | Spruce beetle, mountain pine beetle — often precedes or follows fire |
| **Prescribed Fire** | 6 | Managed burns for fuel reduction / habitat |
| **Vegetation Successional Growth** | 14 | Forest recovery after disturbance |
| **Stable** | 15 | No detected change |

> 🔍 **Look for:** (1) large wildfire spikes, (2) multi-year insect/disease pulses, and (3) whether successional growth follows disturbance years with a lag.

In [ ]:
change_result = cl.summarize_and_chart(
    lcms,
    geometry=study_area,
    band_names='Change',
    scale=120,
    area_format='Percentage',
    title='Uinta Mountains — Annual Change Agent 1985–2024',
    chart_type='line',
    stacked=False,          # unstacked so individual disturbance agents are legible
    date_format='YYYY',
    # Only show the most ecologically relevant classes; others are toggled off by default
    class_visible=[
        'Wildfire',
        'Insect, Disease, or Drought Stress',
        'Prescribed Fire',
        'Vegetation Successional Growth',
        'Stable',
    ],
    width=1000,
    height=550,
)

cl.save_chart_html(change_result['chart'], 'uinta_change_timeseries.html')
print('Chart saved to uinta_change_timeseries.html')
change_result['chart'].show()


## 7 · Analysis — How Has Land Use Shifted? (Sankey Diagram)

The Land Use Sankey diagram shows how area has flowed between use classes across key time periods:
- **1985** → baseline (beginning of LCMS record)
- **2000** → after 15 years; first major beetle outbreak period in the Uintas
- **2012** → after peak spruce beetle epidemic; major fire years
- **2024** → present

The width of each flow is proportional to the area (% of study region) making that transition.

> 💡 In the Uintas, watch for flows between **Forest** and **Other/Rangeland** — these can signal large canopy-loss events that temporarily reclassify forest-use land.

In [ ]:
lu_sankey_result = cl.summarize_and_chart(
    lcms,
    geometry=study_area,
    band_names='Land_Use',
    scale=120,
    area_format='Percentage',
    title='Uinta Mountains — Land Use Transitions 1985 → 2000 → 2012 → 2024',
    sankey=True,
    transition_periods=[1985, 2000, 2012, 2024],  # flat list of transition years
    min_percentage=0.5,    # hide flows < 0.5% to keep the diagram readable
    width=1000,
    height=600,
)

cl.save_chart_html(lu_sankey_result['chart'], 'uinta_land_use_sankey.html')
print('Sankey saved to uinta_land_use_sankey.html')

# Display inline in Jupyter
display(HTML(lu_sankey_result['chart']))


### 7a · Land Use transition matrices

The raw transition numbers behind the Sankey — useful for quantifying how much area moved between classes in each period.

In [ ]:
if 'matrix' in lu_sankey_result:
    for period_key, mat in lu_sankey_result['matrix'].items():
        print(f'### {period_key}')
        print(mat.to_markdown())
        print()


## 8 · Combined Analysis — Forest Cover vs. Disturbance

Now we put it all together: does tree cover in the Uintas track closely with disturbance years? We overlay the annual **Trees** percentage (from Land Cover) against annual **Wildfire** and **Insect/Disease** area (from Change) on a dual-axis chart.

This cell uses `pandas` and `plotly` directly, building on the DataFrames from the previous analyses.

In [ ]:
import plotly.graph_objects as go

# ── Pull relevant columns from the DataFrames computed above ─────────────────
# lc_df: rows = years, columns = Land Cover class names (% of study area)
# change_df: rows = years, columns = Change class names

change_df = change_result['df']

# Guard: check that the expected columns exist before plotting
TREES_COL    = 'Trees'
FIRE_COL     = 'Wildfire'
INSECT_COL   = 'Insect, Disease, or Drought Stress'
GROWTH_COL   = 'Vegetation Successional Growth'

available_lc     = lc_df.columns.tolist()
available_change = change_df.columns.tolist()
print('Land Cover columns :', available_lc)
print('Change columns     :', available_change)

# ── Build dual-axis Plotly figure ────────────────────────────────────────────
fig = go.Figure()

# Left axis — Tree cover (%)
if TREES_COL in lc_df.columns:
    fig.add_trace(go.Scatter(
        x=lc_df.index, y=lc_df[TREES_COL],
        name='Trees (Land Cover %)',
        mode='lines+markers',
        line=dict(color='#004e2b', width=2.5),
        marker=dict(size=4),
        yaxis='y1',
    ))

# Right axis — Disturbance area (%)
for col, color in [(FIRE_COL, '#d54309'), (INSECT_COL, '#f39268'), (GROWTH_COL, '#00a398')]:
    if col in change_df.columns:
        fig.add_trace(go.Bar(
            x=change_df.index, y=change_df[col],
            name=col,
            marker_color=color,
            opacity=0.7,
            yaxis='y2',
        ))

fig.update_layout(
    title='Uinta Mountains — Tree Cover vs. Disturbance Agents 1985–2024',
    xaxis=dict(title='Year', tickmode='linear', dtick=5),
    yaxis=dict(title='Tree Cover (% of study area)', titlefont=dict(color='#004e2b'),
               tickfont=dict(color='#004e2b'), side='left'),
    yaxis2=dict(title='Disturbance / Growth (% of study area)',
                titlefont=dict(color='#d54309'),
                tickfont=dict(color='#d54309'),
                overlaying='y', side='right', showgrid=False),
    barmode='overlay',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    width=1050,
    height=550,
    template='plotly_white',
)

fig.write_html('uinta_combined_analysis.html')
print('Combined chart saved to uinta_combined_analysis.html')
fig.show()


## 9 · Interactive Map — Cumulative Disturbance (All Years)

The time-lapse map adds one layer per year of the LCMS Change band so you can step through time and see *where* fires and insect outbreaks occurred spatially, not just *how much* area they affected.

> ⏱️ This may take 30–60 seconds to load — it is rendering 40 annual layers.

In [ ]:
Map.clearMap()

# Annual LCMS Change time-lapse (slider in the geeViz map)
Map.addTimeLapse(
    lcms.select('Change'),
    {'autoViz': True, 'canAreaChart': True},
    'Change Agent (Annual)',
    visible=True,
)

# Annual Land Cover time-lapse
Map.addTimeLapse(
    lcms.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    'Land Cover (Annual)',
    visible=False,
)

# Study area outline
Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffffff', 'strokeWidth': 2,
     'fillColor': '00000000'},
    'Uinta Mountains Study Area',
)

Map.setCenter(-110.35, 40.65, 9)
Map.view()


## 10 · Bonus — Isolate Pixels with Repeated Disturbance

Some locations in the Uintas have been hit by insects *and* fire — a well-documented interaction where beetle kill creates fuel loads that subsequently burn. This cell flags pixels that experienced both Insect/Disease (class 12) and Wildfire (class 7) at any point in the record.

In [ ]:
# ── Build "ever-experienced" masks for wildfire and insect/disease ────────────
change_ic = lcms.select('Change')

# 1 where each pixel was classified as that agent in ANY year; 0 otherwise
ever_fire   = change_ic.map(lambda img: img.eq(7)).max()   # Wildfire = class 7
ever_insect = change_ic.map(lambda img: img.eq(12)).max()  # Insect/Disease = class 12

# Pixels hit by BOTH at some point in the record
both = ever_fire.And(ever_insect).selfMask().rename('fire_and_insect')

# ── Set class properties so autoViz works ─────────────────────────────────────
both = both.set({
    'fire_and_insect_class_values':  [1],
    'fire_and_insect_class_names':   ['Fire AND Insect/Disease'],
    'fire_and_insect_class_palette': ['ff6600'],
})

# ── Map ───────────────────────────────────────────────────────────────────────
Map.clearMap()

Map.addLayer(ever_fire.selfMask().set({
    'Change_class_values': [1], 'Change_class_names': ['Any Wildfire Year'],
    'Change_class_palette': ['d54309']}),
    {'autoViz': True}, 'Ever Burned (1985–2024)', True)

Map.addLayer(ever_insect.selfMask().set({
    'Change_class_values': [1], 'Change_class_names': ['Any Insect/Disease Year'],
    'Change_class_palette': ['f39268']}),
    {'autoViz': True}, 'Ever Insect/Disease (1985–2024)', True)

Map.addLayer(both, {'autoViz': True}, 'Fire + Insect/Disease (both)', True)

Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffffff', 'strokeWidth': 2,
     'fillColor': '00000000'},
    'Uinta Mountains Study Area',
)

Map.setCenter(-110.35, 40.65, 9)
Map.view()


## 11 · Key Takeaways and Next Steps

### What LCMS tells us about the Uinta Mountains

After running this notebook you should be able to answer:

1. **Forest cover trend** — Has the percentage of tree cover increased, decreased, or remained stable from 1985–2024? Look for long-term decline vs. short-term dips followed by recovery.

2. **Disturbance timing and magnitude** — Which years had the largest wildfire extent? When did insect/disease peak? (Spruce beetle outbreaks in the Uintas intensified around 2000–2015.)

3. **Land use stability** — Does the Sankey show mostly stable Forest-class land, or significant flow into Other/Rangeland following large disturbances?

4. **Fire–insect interaction** — How much of the landscape experienced both insect damage and subsequent fire? Does that co-occurrence cluster spatially?

---

### Going further

| Idea | How |
|------|-----|
| Narrow to a specific sub-watershed or ranger district | Replace `study_area` with `sal.getUSFSDistricts(forest_name='Ashley', district_name='...')` |
| Compare the Uintas to another range | Duplicate cells 5–8 with a different bounding box |
| Overlay elevation / aspect | Add a DEM layer and look for disturbance clustering by elevation band |
| Export annual maps to GeoTIFF | Use `geeViz.geeVizApp.export_image()` or the EE batch export API |
| Add MTBS fire perimeters | Load `FireCCI51` or MTBS from GEE catalog and overlay on the time-lapse map |

---

### Data citation

> USFS GTAC. (2024). *Landscape Change Monitoring System v2024-10*. USDA Forest Service, Geospatial Technology and Applications Center. [https://www.fs.usda.gov/lcms](https://www.fs.usda.gov/lcms)

> Google Earth Engine catalog: `USFS/GTAC/LCMS/v2024-10`
